# 01 — Warehouse population

**Notebook 01 of 7 in the SuSSE refactor track.**

This notebook is the acceptance gate for the warehouse-population step of the refactor: it walks through the data sources SuSSE consumes, the BigQuery layout that holds them, and the three ingest patterns the new `susse.warehouse_ops.population` subsystem supports end-to-end.

Read top-to-bottom. Code cells default to **preview / read-only mode** — no API calls, no warehouse writes — so you can re-run safely without burning quota. Flip the `RUN_INGEST` toggle near the top once you're ready to actually fetch and load.

## Why this notebook exists

The SuSSE project corrects bias in satellite-derived GHI estimates over Sub-Saharan Africa. The model takes a satellite GHI estimate as an input and outputs a corrected GHI; therefore inference at a new (lat, lon, date) **requires** a satellite estimate for that point.

The warehouse plays two distinct roles:

1. **Training corpus** — paired ground-truth and satellite estimates at the 28 ground stations we have (Uganda, Kenya, Ghana, Nigeria, Madagascar, Somalia, Egypt). Used to fit the bias-correction model.
2. **Portal inference cache** — a pre-computed grid of satellite estimates across the region so the portal can answer "what's the corrected GHI at (lat, lon, date)?" in a single BQ lookup, without going live to NASA POWER or CAMS for every user request.

These two roles share the same tables but grow along different axes (more stations and more history vs. denser geographic coverage), and they motivate the three ingest patterns this notebook demonstrates.

## The data sources

### NASA POWER (Prediction Of Worldwide Energy Resources)

Open atmospheric and solar dataset from NASA Langley Research Center. Daily resolution, **0.5° grid (~55 km)** globally. Combines satellite observations and reanalysis. Used here for both the GHI/DHI/DNI "satellite estimate" baseline and a wide set of auxiliary atmospheric variables (temperature, humidity, AOD, cloud cover, etc.) that go into the model as features.

* No authentication required.
* Polite-use throttling — batch requests, don't hammer.
* Lag of about a week from real-time.
* https://power.larc.nasa.gov/

### CAMS Radiation (Copernicus Atmosphere Monitoring Service)

ECMWF-operated solar radiation service via the SoDa platform. Daily and sub-daily resolution, **~5.5 km native grid** (much finer than NASA POWER) over Europe, Africa, Middle East, parts of South America and Atlantic. Returns all-sky and clear-sky GHI/DHI/DNI/BHI plus the McClear clear-sky model.

* Requires registered email at https://www.soda-pro.com/web-services/radiation/cams-radiation-service
* Per-email request quotas; manual ingest, no cron job, until we map the limits.
* Near-real-time availability.

### MERRA-2 (deferred)

NASA's reanalysis dataset (0.5° × 0.625°, hourly). Already wired into `susse.api_clients.merra_2` but not yet ingested into the warehouse. The streaming OPeNDAP path makes it more involved than NASA POWER. Future ingest job.

### MODIS (deferred)

Aqua/Terra satellite products (cloud cover, aerosol, surface reflectance). Higher spatial resolution but more complex retrieval. Future ingest job.

### Ground-truth

The training labels. Currently 25,044 daily measurements across 28 stations from three providers:

* **CrossBoundary (CBE_Data)** — utility-scale stations, Egypt/Ghana/Kenya/Madagascar/Nigeria/Somalia.
* **Makerere University Physics Dept (MAK_physics_dept)** — Kampala / Lira / Tororo, long-running historical Uganda series (some back to 2011).
* **Ugandan Ministry of Energy (ministry_energy_ug)** — Soroti, Wadelai.

All currently arrive in a common CSV schema. New sources (paper datasets, additional ministries) will arrive over time and may have different layouts; the `GroundSourceAdapter` abstraction lets each source contribute its own parser without touching the rest of the pipeline.

## The three ingest patterns

The `susse.warehouse_ops.population` subsystem implements one shared lifecycle (fetch → validate → MERGE-load) across three patterns:

| Pattern | Plan type | Driver | Used for |
|---|---|---|---|
| 1 | `NamedLocationsPlan` | List of `LocationSpec` | Training data quality — dense temporal coverage at known ground stations. |
| 2 | `GridPlan` | `GridSpec(BoundingBox, lat_step, lon_step)` | Portal coverage cache — uniform geographic coverage at native resolution. |
| 3 | `GroundFilePlan` | `GroundSourceAdapter` + file path | New ground-truth measurements arriving from publications/ministries. |

All three patterns are **idempotent**: each job pre-checks the warehouse for existing keys (date, geohash, variable, source) and skips fetching what's already cached. Re-running a job is cheap (one BQ scan) instead of expensive (re-hitting the API). Manual invocation only — no cron, no automatic retry-with-backoff, until we map empirical rate-limit behaviour.

## Setup

In [1]:
from datetime import date
from pathlib import Path
import logging
import os

import pandas as pd

from susse.warehouse_ops.io import (
    BigQueryClient,
    TableRefs,
    TableSchemas,
    WarehouseConfig,
)
from susse.warehouse_ops.population import (
    BoundingBox,
    CamsSatelliteJob,
    CurationOptions,
    DateRange,
    GridPlan,
    GridSpec,
    GroundFilePlan,
    GroundIngestJob,
    LocationSpec,
    NamedLocationsPlan,
    NasaPowerSatelliteJob,
    Source,
    StandardCsvAdapter,
    VariableCatalog,
    curate_ground,
    populate_dim_variable,
    variables_to_dataframe,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

In [2]:
# Master toggle: when False, the notebook reads the warehouse and shows what
# would happen, but never calls a satellite API or writes a row. Flip to True
# only when you actually want to ingest.
RUN_INGEST = False

config = WarehouseConfig()  # defaults to solar-irradiation-estimation / solar_warehouse
tables = TableRefs(config=config)
bq = BigQueryClient(config=config)

print(f"project : {config.project_id}")
print(f"dataset : {config.dataset}")
print(f"RUN_INGEST = {RUN_INGEST}")

project : solar-irradiation-estimation
dataset : solar_warehouse
RUN_INGEST = False


## Live warehouse tour

What's in the warehouse right now, queried directly. Re-running this notebook over time will show the warehouse growing as we run the ingest jobs.

In [3]:
tables_df = bq.query(f"""
SELECT table_id, row_count, ROUND(size_bytes / 1024 / 1024, 1) AS size_mb
FROM `{config.project_id}.{config.dataset}.__TABLES__`
ORDER BY size_bytes DESC
""")
tables_df

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_id,row_count,size_mb
0,nasa_daily_vars_long,20789532,2048.1
1,irradiance_daily,1436184,143.8
2,ground_measurements,25044,3.3
3,ground_measurements_raw,25044,1.1
4,dim_variable,4,0.0
5,cams_daily_ext,0,0.0
6,nasa_daily_ext,0,0.0
7,v_irr_monthly_by_point,0,0.0
8,v_irr_weekly_by_point,0,0.0


In [4]:
irradiance_summary = bq.query(f"""
SELECT source,
       COUNT(*)                            AS n_rows,
       COUNT(DISTINCT geohash5)            AS n_points,
       MIN(date)                           AS first_date,
       MAX(date)                           AS last_date,
       ROUND(MIN(latitude), 2)             AS min_lat,
       ROUND(MAX(latitude), 2)             AS max_lat,
       ROUND(MIN(longitude), 2)            AS min_lon,
       ROUND(MAX(longitude), 2)            AS max_lon
FROM `{tables.irradiance_daily}`
GROUP BY source
ORDER BY source
""")
irradiance_summary

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,source,n_rows,n_points,first_date,last_date,min_lat,max_lat,min_lon,max_lon
0,CAMS,718092,1962,2024-01-01,2024-12-31,-1.38,4.22,29.67,34.97
1,NASA,718092,1962,2024-01-01,2024-12-31,-1.38,4.22,29.67,34.97


In [5]:
ground_summary = bq.query(f"""
SELECT location,
       COUNT(*)                AS n_days,
       MIN(date)               AS first_date,
       MAX(date)               AS last_date,
       ROUND(AVG(lat), 4)      AS lat,
       ROUND(AVG(lon), 4)      AS lon,
       ROUND(AVG(ghi_kwh_m2_day), 2) AS ghi_mean_kwh
FROM `{tables.ground_measurements}`
GROUP BY location
ORDER BY n_days DESC
""")
ground_summary

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,location,n_days,first_date,last_date,lat,lon,ghi_mean_kwh
0,kenya_location3,1976,2019-05-28,2024-11-25,-0.4700,35.1819,5.89
1,lira,1935,2014-08-27,2023-02-01,2.2952,32.9214,4.43
2,kampala,1798,2011-04-06,2023-01-22,0.3335,32.5686,4.49
3,ghana_location1,1774,2019-09-24,2024-11-25,5.6458,-0.1052,4.69
4,nigeria_location1,1641,2020-04-04,2024-11-25,9.0764,7.4254,4.95
5,kenya_location5,1382,2020-12-22,2024-11-25,-0.2200,35.8600,5.67
6,tororo,1380,2011-03-08,2017-11-22,0.6978,34.1715,5.66
7,ghana_location3,1320,2021-02-25,2024-11-25,-0.2351,5.6322,4.29
8,ghana_location2,1105,2020-01-31,2024-05-02,5.6457,0.0087,4.87
9,kenya_location2,1016,2019-10-08,2024-02-23,0.6100,36.8000,5.92


## Pattern 1 — Named-location satellite ingest

Fetch a satellite source for a discrete set of locations — typically the ground stations whose history we want covered for training.

We'll build a plan for **Uganda 2025 + Kenya 2024** at the existing ground-station coordinates and let the job decide which slices need fetching. If the warehouse already has the data, the coverage check returns it as fully cached and skips the API call.

In [6]:
# Pull station coordinates from the warehouse rather than hardcoding them.
stations = bq.query(f"""
SELECT location, ROUND(AVG(lat), 6) AS lat, ROUND(AVG(lon), 6) AS lon
FROM `{tables.ground_measurements}`
WHERE STARTS_WITH(location, 'kenya_') OR location IN ('kampala','lira','tororo','soroti','wadelai')
GROUP BY location
ORDER BY location
""")
stations

/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,location,lat,lon
0,kampala,0.333542,32.568630
1,kenya_location10,-1.451111,36.975687
2,kenya_location11,-1.494448,37.057094
3,kenya_location12,-0.224000,35.880000
4,kenya_location13,-1.491302,37.052862
5,kenya_location14,-1.451315,36.973008
6,kenya_location2,0.610000,36.800000
7,kenya_location3,-0.469974,35.181874
8,kenya_location4,-1.232761,36.878509
9,kenya_location5,-0.220000,35.860000


In [7]:
uganda_locations = tuple(
    LocationSpec(name=row.location, lat=float(row.lat), lon=float(row.lon))
    for row in stations.itertuples(index=False)
    if row.location in {"kampala", "lira", "tororo", "soroti", "wadelai"}
)
kenya_locations = tuple(
    LocationSpec(name=row.location, lat=float(row.lat), lon=float(row.lon))
    for row in stations.itertuples(index=False)
    if row.location.startswith("kenya_")
)

uganda_2025 = DateRange(start=date(2025, 1, 1), end=date(2025, 1, 31))
kenya_2024  = DateRange(start=date(2024, 1, 1), end=date(2024, 1, 31))

# Fetch the full set of NASA POWER variables this project tracks.
nasa_vars = VariableCatalog.for_source(Source.NASA_POWER)
cams_vars = VariableCatalog.for_source(Source.CAMS)

uganda_nasa_plan = NamedLocationsPlan(
    source=Source.NASA_POWER,
    date_range=uganda_2025,
    locations=uganda_locations,
    variables=nasa_vars,
)
kenya_nasa_plan = NamedLocationsPlan(
    source=Source.NASA_POWER,
    date_range=kenya_2024,
    locations=kenya_locations,
    variables=nasa_vars,
)
uganda_cams_plan = NamedLocationsPlan(
    source=Source.CAMS,
    date_range=uganda_2025,
    locations=uganda_locations,
    variables=cams_vars,
)
kenya_cams_plan = NamedLocationsPlan(
    source=Source.CAMS,
    date_range=kenya_2024,
    locations=kenya_locations,
    variables=cams_vars,
)

for plan in (uganda_nasa_plan, kenya_nasa_plan, uganda_cams_plan, kenya_cams_plan):
    print(plan.describe())

NamedLocations[source=NASA_POWER, locations=5, vars=32, dates=2025-01-01..2025-01-31]
NamedLocations[source=NASA_POWER, locations=13, vars=32, dates=2024-01-01..2024-01-31]
NamedLocations[source=CAMS, locations=5, vars=9, dates=2025-01-01..2025-01-31]
NamedLocations[source=CAMS, locations=13, vars=9, dates=2024-01-01..2024-01-31]


In [8]:
if RUN_INGEST:
    nasa_job = NasaPowerSatelliteJob(bq=bq, refs=tables)
    for plan in (uganda_nasa_plan, kenya_nasa_plan):
        result = nasa_job.run(plan)
        print(result.summary())
else:
    print("RUN_INGEST=False — skipping NASA POWER named-location runs.")
    print("Each plan above describes one job invocation; flip RUN_INGEST to execute.")

RUN_INGEST=False — skipping NASA POWER named-location runs.
Each plan above describes one job invocation; flip RUN_INGEST to execute.


In [9]:
# CAMS requires a registered email. Set CAMS_EMAIL in your environment
# (or .env) before running this cell.
if RUN_INGEST and os.getenv("CAMS_EMAIL"):
    cams_job = CamsSatelliteJob(bq=bq, refs=tables)
    for plan in (uganda_cams_plan, kenya_cams_plan):
        result = cams_job.run(plan)
        print(result.summary())
elif RUN_INGEST:
    print("RUN_INGEST=True but CAMS_EMAIL is not set — skipping CAMS runs.")
    print("Register an email at https://www.soda-pro.com/ and put it in .env as CAMS_EMAIL.")
else:
    print("RUN_INGEST=False — skipping CAMS named-location runs.")

RUN_INGEST=False — skipping CAMS named-location runs.


## Pattern 2 — Grid satellite ingest

The portal needs a satellite GHI estimate for any (lat, lon) in the region a user might query. Pre-cache them with a `GridPlan` over a bounding box at the source's native resolution, so the portal can serve cached estimates directly.

Below: a tiny demo grid over a 0.5° × 0.5° patch of central Uganda at CAMS native 0.05° resolution — 121 grid points. Real region-wide grids will be much larger and run as separate, deliberate batches; the demo is here to exercise the API.

In [10]:
uganda_demo_bbox = BoundingBox(
    min_lat=0.10, max_lat=0.60,
    min_lon=32.30, max_lon=32.80,
)
demo_grid = GridSpec(bbox=uganda_demo_bbox, lat_step=0.05, lon_step=0.05)
demo_dates = DateRange(start=date(2025, 1, 1), end=date(2025, 1, 7))

demo_grid_plan = GridPlan(
    source=Source.CAMS,
    date_range=demo_dates,
    grid=demo_grid,
    variables=cams_vars,
)
print(demo_grid_plan.describe())
print(f"Grid points: {demo_grid.n_points}")

Grid[source=CAMS, points=121, vars=9, dates=2025-01-01..2025-01-07]
Grid points: 121


In [11]:
if RUN_INGEST and os.getenv("CAMS_EMAIL"):
    cams_job = CamsSatelliteJob(bq=bq, refs=tables)
    grid_result = cams_job.run(demo_grid_plan)
    print(grid_result.summary())
else:
    print("Grid ingest skipped (RUN_INGEST=False or CAMS_EMAIL not set).")
    print("This plan would call CAMS once per grid point that isn't already cached.")

Grid ingest skipped (RUN_INGEST=False or CAMS_EMAIL not set).
This plan would call CAMS once per grid point that isn't already cached.


## Pattern 3 — Ground-truth ingest

Future ground-truth additions will arrive in heterogeneous formats from publications, ministries, and partner organisations. The `GroundSourceAdapter` interface lets each new source contribute its own parser without touching the rest of the pipeline.

The CrossBoundary, Makerere, and Ministry-of-Energy CSVs all share the standard `(datetime, ghi, location, latitude, longitude)` schema, so a single `StandardCsvAdapter` handles all three. When a source with a different layout arrives, write a new `GroundSourceAdapter` subclass.

This demo runs end-to-end against the real CrossBoundary Somalia file. Because every row is already in `ground_measurements_raw` from the existing warehouse population, the idempotent coverage check causes the job to add zero rows — a real safety check that the pipeline doesn't double-write.

In [12]:
# Real on-disk file from the project's data directory.
somalia_csv = Path("../data/ground_measurements/CBE_Data/somalia.csv").resolve()
assert somalia_csv.exists(), f"Expected {somalia_csv} to exist."

adapter = StandardCsvAdapter(source_id="CBE")
raw_df = adapter.parse(somalia_csv)
print(f"Parsed {len(raw_df)} raw rows.")
raw_df.head()

Parsed 296 raw rows.


,datetime,ghi,location,latitude,longitude
0,2024-01-28,5550.759,somalia_location1,3.107191,43.637153
1,2024-02-01,5609.558,somalia_location1,3.107191,43.637153
2,2024-02-02,6353.845,somalia_location1,3.107191,43.637153
3,2024-02-03,5800.749,somalia_location1,3.107191,43.637153
4,2024-02-04,6444.199,somalia_location1,3.107191,43.637153


In [13]:
# Curation is a pure transformation — no I/O. Run it on the parsed raw rows
# to see what would land in `ground_measurements`.
curated_df = curate_ground(
    raw_df,
    CurationOptions(qc_level="auto", version="v1", geohash_precision=5),
)
print("Curated columns:", list(curated_df.columns))
curated_df.head()

Curated columns: ['date', 'month', 'location', 'lat', 'lon', 'geohash5', 'ghi_wh_m2_day', 'ghi_kwh_m2_day', 'qc_level', '_version', '_curated_at']


,date,month,location,lat,lon,geohash5,ghi_wh_m2_day,ghi_kwh_m2_day,qc_level,_version,_curated_at
0,2024-01-28,2024-01-01,somalia_location1,3.107191,43.637153,sbx18,5550.759,5.550759,auto,v1,2026-05-08 08:36:36.160269+00:00
1,2024-02-01,2024-02-01,somalia_location1,3.107191,43.637153,sbx18,5609.558,5.609558,auto,v1,2026-05-08 08:36:36.160269+00:00
2,2024-02-02,2024-02-01,somalia_location1,3.107191,43.637153,sbx18,6353.845,6.353845,auto,v1,2026-05-08 08:36:36.160269+00:00
3,2024-02-03,2024-02-01,somalia_location1,3.107191,43.637153,sbx18,5800.749,5.800749,auto,v1,2026-05-08 08:36:36.160269+00:00
4,2024-02-04,2024-02-01,somalia_location1,3.107191,43.637153,sbx18,6444.199,6.444199,auto,v1,2026-05-08 08:36:36.160269+00:00


In [14]:
# Run the full ingest path. Idempotent: if the data is already in the warehouse
# (which it is, for this CSV), the job adds zero rows and reports them all as
# already cached.
if RUN_INGEST:
    ground_plan = GroundFilePlan(
        source_id="CBE",
        file_path=somalia_csv,
        adapter_id=adapter.adapter_id,
    )
    ground_job = GroundIngestJob(bq=bq, adapter=adapter, refs=tables)
    ground_result = ground_job.run(ground_plan)
    print(ground_result.summary())
else:
    print("Ground ingest skipped (RUN_INGEST=False).")

Ground ingest skipped (RUN_INGEST=False).


## Long-format CAMS twin

We add a `cams_daily_vars_long` table mirroring `nasa_daily_vars_long` so non-irradiance CAMS variables (BHI, clear-sky components, extraterrestrial irradiance) live in a uniform long format. The DDL lives in `warehouse/sql/00_schema/cams_daily_vars_long.sql` and is `CREATE TABLE IF NOT EXISTS` — safe to re-run.

In [15]:
ddl_path = Path("../warehouse/sql/00_schema/cams_daily_vars_long.sql").resolve()
ddl_sql = ddl_path.read_text()
print(ddl_sql)

if RUN_INGEST:
    bq.execute_ddl(ddl_sql)
    print("cams_daily_vars_long table ensured.")
else:
    print("DDL execution skipped (RUN_INGEST=False).")

-- Long-format daily CAMS variables.
-- Mirrors nasa_daily_vars_long: one row per (date, point, variable, source).
-- Created idempotently so this DDL is safe to re-run during deployment.

CREATE TABLE IF NOT EXISTS `solar-irradiation-estimation.solar_warehouse.cams_daily_vars_long` (
    date         DATE,
    latitude     FLOAT64,
    longitude    FLOAT64,
    geog         GEOGRAPHY,
    geohash5     STRING,
    variable_id  STRING,
    value        FLOAT64,
    source       STRING
)
PARTITION BY date
CLUSTER BY geohash5, variable_id;

DDL execution skipped (RUN_INGEST=False).


## `dim_variable` backfill

The existing warehouse has only 4 rows in `dim_variable` despite 27+ distinct `variable_id`s in `nasa_daily_vars_long`, and the existing rows have `spatial_resolution_km = -51.0` (a sign-error sentinel bug). The `VariableCatalog` registers all 41 variables (32 NASA POWER + 9 CAMS) with correct positive resolutions; `populate_dim_variable` MERGE-upserts them by `(variable_id, source)`.

In [16]:
current_dim = bq.query(f"SELECT * FROM `{tables.dim_variable}` ORDER BY source, variable_id")
print(f"Current dim_variable rows: {len(current_dim)}")
current_dim

Current dim_variable rows: 4


/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,variable_id,source,display_name,unit,native_unit,description,temporal_granularity,spatial_resolution_km,valid_min,valid_max
0,aod_550,NASA,Aerosol Optical Depth @550nm,unitless,unitless,Column AOD at 550 nm,daily,-51.0,NaN,NaN
1,relative_humidity,NASA,Relative Humidity,%,%,Daily relative humidity,daily,-51.0,NaN,NaN
2,surface_pressure,NASA,Surface Pressure,hPa,hPa,Surface level pressure,daily,-51.0,NaN,NaN
3,temperature,NASA,2m Air Temperature,degC,degC,Daily near-surface air temperature,daily,-51.0,NaN,NaN


In [17]:
catalog_df = variables_to_dataframe(VariableCatalog.all_variables())
print(f"Catalog rows ready to upsert: {len(catalog_df)}")
catalog_df.head(10)

Catalog rows ready to upsert: 41


,variable_id,source,display_name,unit,native_unit,description,temporal_granularity,spatial_resolution_km,valid_min,valid_max
0,ghi,NASA_POWER,All-sky GHI,kWh/m^2/day,kWh/m^2/day,All-sky surface shortwave downward irradiance.,daily,55.5,NaN,NaN
1,dhi,NASA_POWER,All-sky DHI,kWh/m^2/day,kWh/m^2/day,All-sky diffuse horizontal irradiance.,daily,55.5,NaN,NaN
2,dni,NASA_POWER,All-sky DNI,kWh/m^2/day,kWh/m^2/day,All-sky direct normal irradiance.,daily,55.5,NaN,NaN
3,temperature,NASA_POWER,2m Air Temperature,degC,degC,Daily near-surface air temperature.,daily,55.5,NaN,NaN
4,temperature_range,NASA_POWER,2m Air Temperature Range,degC,degC,Daily range (max - min) of 2m air temperature.,daily,55.5,NaN,NaN
5,specific_humidity,NASA_POWER,2m Specific Humidity,kg/kg,kg/kg,2m specific humidity.,daily,55.5,NaN,NaN
6,relative_humidity,NASA_POWER,2m Relative Humidity,%,%,2m relative humidity.,daily,55.5,NaN,NaN
7,surface_pressure,NASA_POWER,Surface Pressure,kPa,kPa,Daily surface-level pressure.,daily,55.5,NaN,NaN
8,aod_550,NASA_POWER,AOD @ 550nm,unitless,unitless,Aerosol Optical Depth at 550 nm.,daily,55.5,NaN,NaN
9,aod_550_adj,NASA_POWER,AOD @ 550nm (adjusted),unitless,unitless,"Aerosol Optical Depth at 550 nm, terrain-adjus...",daily,55.5,NaN,NaN


In [18]:
if RUN_INGEST:
    n_written = populate_dim_variable(bq, table_fqn=tables.dim_variable)
    print(f"Wrote {n_written} dim_variable rows.")
    after = bq.query(f"""
    SELECT source, COUNT(*) AS n_vars, COUNT(DISTINCT spatial_resolution_km) AS n_resolutions,
           MIN(spatial_resolution_km) AS min_res_km, MAX(spatial_resolution_km) AS max_res_km
    FROM `{tables.dim_variable}`
    GROUP BY source
    """)
    print(after)
else:
    print("dim_variable backfill skipped (RUN_INGEST=False).")

dim_variable backfill skipped (RUN_INGEST=False).


## What's next

With the population subsystem in place, several growth axes open up that we are deliberately deferring:

* **Continental grid** — expand pattern 2 to a Sub-Saharan-Africa-wide grid (~16K NASA POWER points at 0.5°; ~1.6M CAMS points at 0.05°). Treat it as several deliberate batches, watch the API quotas empirically before automating.
* **Scheduling** — once we understand the rate-limit landscape, a periodic job to keep the warehouse caught up with NASA POWER (~1 week lag) and CAMS (near-real-time).
* **Portal-side fallback** — if the portal queries a (lat, lon, date) the warehouse doesn't yet cover, fall back to a live API call rather than fail. Graceful-degradation only.
* **MERRA-2 and MODIS jobs** — already have API clients in `susse.api_clients`; need new `BaseSatelliteJob` subclasses to plug them in.
* **QC formalisation** — the curation function currently stamps `qc_level="auto"` on every row. Real QC rules (gap detection, outlier flags, missing-data thresholds) belong in the curation function as new sources arrive.
* **Adapters for new sources** — the next ground-truth dataset likely won't share the standard CSV schema. Each new source gets a `GroundSourceAdapter` subclass; nothing else in the pipeline changes.

Notebook 02 will pick up where this leaves off, building the `TrainingDataset` snapshot that the model layer consumes.